In [5]:
from google.colab import userdata
from google import genai
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Client ready ✅")

Client ready ✅


In [18]:
import json, time, hashlib
from typing import Optional
from pydantic import BaseModel, Field, ValidationError
from google.genai import types

MODEL = None      # pick_working_model() (neeche wali cell) set karegi      # wahi naam jo pehle chala tha
CACHE_PATH = "llm_cache.json"

In [7]:
import pandas as pd

COLUMNS = [
    "PLAN TYPE", "PRODUCT NAME", "BRAND", "VARIANT", "BASE", "MSRP",
    "RESELLER LIST PRICE", "SWITCHER", "SWITCHER_OFFER_TEXT", "UNIQUE DISCOUNT",
    "FREE MEMORY UPGRADE", "BOGO OFFER", "Max TI", "Trade-In Value Type",
    "LINE TYPE", "RATEPLAN DURATION (MONTHS)", "SHIPMENT", "STOCK",
    "PWP/GWP OFFERS", "Cleaned PWP/GWP OFFERS", "PWP/GWP OFFER $ VALUE",
    "SUBSCRIPTION BUNDLE OFFERS", "SUBSCRIPTION BUNDLE OFFER $ VALUE",
    "SERVICE LINE OFFERS", "SERVICE LINE OFFER $ VALUE",
    "ADDITIONAL OFFERS", "ADDITIONAL OFFER $ VALUE",
    "FINANCE OFFER DESCRIPTION 1", "FINANCE OFFER $ VALUE",
]
LINE_TYPES = ["New Line", "Upgrade"]

with open("Input.json") as f:
    products = json.load(f)

print(len(products), "products loaded")    # 12 aana chahiye

12 products loaded


In [8]:
def offer_to_text(offer):
    """Ek offer ke saare text hisse (title, description, modal, card legal) jod do."""
    parts = [offer.get("title"), offer.get("description")]
    details = offer.get("full_details") or {}
    parts.append(details.get("modal_raw_text"))
    for card in details.get("cards") or []:
        parts.append(card.get("legal"))
    return "\n".join(p.strip() for p in parts if p)

def text_id(text):
    """Text ka chhota fingerprint, taaki same text dobara LLM ko na bheje."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:10]

unique_texts = {}          # {id: text}
product_offer_ids = []     # har product ke offers ke ids, products ke order mein
for p in products:
    ids = []
    for offer in p.get("offers") or []:
        text = offer_to_text(offer)
        tid = text_id(text)
        unique_texts[tid] = text
        ids.append(tid)
    product_offer_ids.append(ids)

print("Total offers:", sum(len(x) for x in product_offer_ids))
print("Unique texts:", len(unique_texts))
for tid, t in unique_texts.items():
    print(tid, "|", t.split("\n")[0][:70])

Total offers: 48
Unique texts: 11
360e497186 | Includes additional $5.56/mo. bill credit for 36 mos.
ceafcbf1b4 | Get iPhone 17 Pro Max 256GB for $9.43/mo.
17a22dbf6c | Get up to $800/line to break your contract.
e6c0b8cc2a | Earn a $200 statement credit
2662442544 | Includes additional $5.56/mo. bill credit for 36 mos.
6558fc12ce | Get iPhone 17 Pro 256GB for $4.43/mo.
c19d8d2879 | Get iPhone 17 for $0
1521992849 | Online only. Get $200 off a new line.
962ff59b0d | Includes additional $5.56/mo. bill credit for 36 mos.
6f654c6588 | Get iPhone 17e 256GB for $0
b411cd8f86 | Get the new iPhone 18 Pro Max for up to $1,200 off


In [9]:
class Amount(BaseModel):
    value: str = Field(description="Dollar amount copied EXACTLY as written in the text, e.g. '$660.36' or '$1,200'. Never compute or round.")
    variant: Optional[str] = Field(description="Storage variant this amount is tied to, e.g. '256GB' or '1TB'. null if it applies to every variant.")
    kind: str = Field(description="One of: device_discount, service_line_credit, trade_in_max, switcher, finance_one_time, finance_recurring, finance_bill_discount, pwp_gwp, subscription, other")
    is_monthly: bool = Field(description="true only if the amount is a per-month figure like '$5.56/mo.'")

class OfferExtraction(BaseModel):
    category: str = Field(description="One of: device_discount, service_line, trade_in, switcher, finance, pwp_gwp, subscription, other")
    eligible_line_types: list[str] = Field(description="Subset of ['New Line', 'Upgrade'] for which this offer is valid")
    rateplan_months: Optional[int] = Field(description="Installment / credit duration in months if explicitly stated, else null")
    trade_in_value_type: Optional[str] = Field(description="'Enhanced' or 'Base' for trade-in offers, else null")
    amounts: list[Amount] = Field(description="Every relevant dollar amount in the offer (see rules)")
    cleaned_text: str = Field(description="One short sentence summarising the offer using only facts present in the text")

In [10]:
PROMPT_TEMPLATE = """You are a data extraction engine for telecom promotions.
Read the OFFER TEXT and fill the JSON schema. Use ONLY facts written in the text.
Never use outside knowledge, never guess, never calculate, never round.

RULES
1. Amounts: copy dollar values exactly as written (keep decimals and commas).
   Do NOT extract: monthly device prices (e.g. $9.43/mo.), plan prices ($60/mo.),
   activation fees, retailer-partner/business prices, spend requirements
   (e.g. "after spending $1,000"), APRs or card fees.
2. Variant tables: if different storage sizes get different amounts, output one
   Amount per size with its variant (e.g. "256GB").
3. Line types:
   - "new line", "new customers", "port-in", "switch to AT&T" -> ["New Line"]
   - "pick or upgrade to an eligible plan", or no line restriction -> ["New Line", "Upgrade"]
4. Kinds:
   - device bill credits tied to a device/variant -> device_discount
   - "additional $X in credits" for new lines / "$X off a new line" -> service_line_credit
     (use the total like "$200", mark the per-month figure is_monthly=true)
   - "up to $X off" that requires a trade-in -> trade_in_max (only if a dollar max is stated)
   - carrier-switch reward ("up to $800/line") -> switcher
   - third-party credit card: welcome/acquisition bonus -> finance_one_time,
     yearly statement credits -> finance_recurring, "$X discounts on your bills" -> finance_bill_discount
5. trade_in_value_type: "Enhanced" for promotional trade-in deals paid via bill credits,
   "Base" for a standard trade-in program with no promotional amount, else null.
6. If something is not stated, use null (or an empty list). Do not invent.

EXAMPLE
Text: "Eligibility: New customers activating new line. iPhone 17 Pro Max 256GB ... after up to
$660.36 in bill credits ... Price after additional $200 in credits ($5.56/mo.) for new consumer lines only."
-> category device_discount, eligible_line_types ["New Line"],
   amounts: {{$660.36, variant 256GB, device_discount}}, {{$200, null, service_line_credit}},
            {{$5.56, null, service_line_credit, is_monthly true}}

OFFER TEXT
{offer_text}
"""

In [19]:
class DailyQuotaExhausted(Exception):
    """Model ki aaj ki free limit khatam; retry karne ka koi fayda nahi."""
    pass

def extract_offer(text, max_retries=3):
    """Ek offer text ko LLM se structured OfferExtraction mein badlo. Fail ho to None."""
    prompt = PROMPT_TEMPLATE.format(offer_text=text)
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,                          # har baar same, predictable answer
                    response_mime_type="application/json",
                    response_schema=OfferExtraction,
                ),
            )
            if response.parsed is not None:
                return response.parsed
            return OfferExtraction.model_validate_json(response.text)   # backup parse
        except (ValidationError, ValueError) as e:
            print(f"   attempt {attempt}: malformed output -> {str(e)[:200]}")
        except Exception as e:
            msg = str(e)
            if "RESOURCE_EXHAUSTED" in msg or "429" in msg:
                if "PerDay" in msg:
                    raise DailyQuotaExhausted(MODEL) from e      # retry bekaar, model badlo
                print(f"   attempt {attempt}: rate limited, waiting 60s")
                time.sleep(60)
                continue
            print(f"   attempt {attempt}: API error -> {msg[:200]}")   # 503, timeout, network
        time.sleep(2 ** attempt)                                       # 2s, 4s, 8s ruko
    return None

In [20]:
CANDIDATE_MODELS = [
    "models/gemini-3.7-flash",
    "models/gemini-3.6-flash",
    "models/gemini-3.5-flash",
    "models/gemini-3.5-flash-lite",
]

def pick_working_model(candidates):
    """List mein se pehla model jo abhi quota mein ho, wo lauta do."""
    for name in candidates:
        try:
            client.models.generate_content(model=name, contents="hi")
            print("✅ Using", name)
            return name
        except Exception as e:
            reason = "daily quota" if "PerDay" in str(e) else str(e)[:80]
            print("❌", name, "->", reason)
    return None

MODEL = pick_working_model(CANDIDATE_MODELS)

❌ models/gemini-3.7-flash -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
✅ Using models/gemini-3.6-flash


In [12]:
test = extract_offer(unique_texts["360e497186"])
if test:
    print("category:", test.category, "| lines:", test.eligible_line_types,
          "| months:", test.rateplan_months)
    for a in test.amounts:
        print(f"  {a.value:>10} | {a.kind:<22} | variant={a.variant} | monthly={a.is_monthly}")
else:
    print("Extraction failed")

   attempt 1: API error -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
category: device_discount | lines: ['New Line'] | months: 36
     $660.36 | device_discount        | variant=256GB | monthly=False
     $644.36 | device_discount        | variant=512GB | monthly=False
     $628.36 | device_discount        | variant=1TB | monthly=False
     $812.36 | device_discount        | variant=2TB | monthly=False
       $5.56 | service_line_credit    | variant=None | monthly=True


In [23]:
try:
    with open(CACHE_PATH) as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}

if MODEL is None:
    raise RuntimeError("Koi bhi model available nahi hai. Kal dobara chalao.")

def models_after(current):
    """CANDIDATE_MODELS mein current ke baad wale models."""
    return CANDIDATE_MODELS[CANDIDATE_MODELS.index(current) + 1:]

for tid, text in unique_texts.items():
    if cache.get(tid):
        continue                                   # pehle se ho chuka
    print("Extracting", tid, "|", text.split("\n")[0][:60], "| model:", MODEL)
    while True:
        try:
            result = extract_offer(text)
            break
        except DailyQuotaExhausted:
            print("   daily quota over for", MODEL, "-> switching model")
            MODEL = pick_working_model(models_after(MODEL))
            if MODEL is None:
                raise RuntimeError("Saare models ka aaj ka quota khatam. Cache mein kaam save hai.")
    cache[tid] = result.model_dump() if result else None
    with open(CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)
    time.sleep(10)

print("Done:", sum(1 for v in cache.values() if v), "/", len(unique_texts), "offers extracted")

Extracting 6558fc12ce | Get iPhone 17 Pro 256GB for $4.43/mo. | model: models/gemini-3.6-flash
Done: 11 / 11 offers extracted


In [24]:
ALLOWED_KINDS = {"device_discount", "service_line_credit", "trade_in_max", "switcher",
                 "finance_one_time", "finance_recurring", "finance_bill_discount",
                 "pwp_gwp", "subscription", "other"}

def norm(s):
    return s.replace(",", "").replace(" ", "")

def amount_in_text(value, text):
    """LLM ka diya amount source text mein sach mein likha hai ya nahi?"""
    return norm(value) in norm(text)

for tid, ext in cache.items():
    text = unique_texts.get(tid, "")
    print("\n==", tid, "|", text.split("\n")[0][:60])
    if ext is None:
        print("   !! extraction failed")
        continue
    print("   category:", ext["category"], "| lines:", ext["eligible_line_types"],
          "| months:", ext["rateplan_months"], "| TI type:", ext["trade_in_value_type"])
    for a in ext["amounts"]:
        flags = []
        if not amount_in_text(a["value"], text): flags.append("NOT IN TEXT")
        if a["kind"] not in ALLOWED_KINDS:       flags.append("BAD KIND")
        print(f"   {a['value']:>10} | {a['kind']:<22} | variant={a['variant']} "
              f"| monthly={a['is_monthly']} | {'OK' if not flags else '!! ' + ', '.join(flags)}")


== 360e497186 | Includes additional $5.56/mo. bill credit for 36 mos.
   category: device_discount | lines: ['New Line'] | months: 36 | TI type: Base
      $660.36 | device_discount        | variant=256GB | monthly=False | OK
      $644.36 | device_discount        | variant=512GB | monthly=False | OK
      $628.36 | device_discount        | variant=1TB | monthly=False | OK
      $812.36 | device_discount        | variant=2TB | monthly=False | OK
        $5.56 | service_line_credit    | variant=None | monthly=True | OK

== ceafcbf1b4 | Get iPhone 17 Pro Max 256GB for $9.43/mo.
   category: device_discount | lines: ['New Line'] | months: 36 | TI type: None
         $200 | service_line_credit    | variant=None | monthly=False | OK
    $5.56/mo. | service_line_credit    | variant=None | monthly=True | OK

== 17a22dbf6c | Get up to $800/line to break your contract.
   category: switcher | lines: ['New Line'] | months: 36 | TI type: None
         $800 | switcher               | variant=None

In [26]:
import re
from decimal import Decimal

LINE_TYPE_ALIASES = {"new line": "New Line", "upgrade": "Upgrade"}

def norm_variant(v):
    # "256 GB" aur "256GB" ko same banane ke liye
    return re.sub(r"\s+", "", v or "").upper()

def to_decimal(value):
    # float nahi use kiya taaki decimals round na ho
    return Decimal(re.sub(r"[^\d.]", "", value))

def fmt_money(value):
    return "$" + str(to_decimal(value))

def money_sum(values):
    return "$" + str(sum((to_decimal(v) for v in values), Decimal("0")))

def clean_extraction(tid, ext):
    # LLM output check: jo amount text mein nahi hai wo hata do
    if not ext:
        return None
    text = unique_texts.get(tid, "")
    lines = {LINE_TYPE_ALIASES[l.strip().lower()] for l in ext.get("eligible_line_types") or []
             if l.strip().lower() in LINE_TYPE_ALIASES}
    amounts = []
    for a in ext.get("amounts") or []:
        if a.get("kind") not in ALLOWED_KINDS or not amount_in_text(a["value"], text):
            print(f"   dropped {a.get('value')} from {tid}")
            continue
        amounts.append(a)
    return {**ext, "eligible_line_types": lines, "amounts": amounts}

def first_amount(offers, kind, variant=None, allow_monthly=False):
    for tid, ext in offers:
        for a in ext["amounts"]:
            if a["kind"] != kind:
                continue
            if a["is_monthly"] and not allow_monthly:
                continue
            if variant and a.get("variant") and norm_variant(a["variant"]) != variant:
                continue
            return tid, ext, a
    return None

def get_months(product, offers):
    for _, ext in offers:
        if ext["category"] in ("device_discount", "service_line", "trade_in") and ext.get("rateplan_months"):
            return ext["rateplan_months"]
    for opt in product.get("pricing", {}).get("options") or []:
        m = re.search(r"over (\d+) mos", opt)
        if m:
            return int(m.group(1))
    return None

def build_row(product, offer_ids, line_type):
    row = {col: None for col in COLUMNS}
    variant = norm_variant(product.get("variant"))
    pricing = product.get("pricing") or {}

    # seedhe fields
    row["PLAN TYPE"] = product.get("plan_type")
    row["PRODUCT NAME"] = product.get("product_name")
    row["BRAND"] = product.get("brand")
    row["VARIANT"] = product.get("variant")
    row["BASE"] = pricing.get("retail")
    row["MSRP"] = pricing.get("retail")
    row["LINE TYPE"] = line_type
    row["SHIPMENT"] = product.get("shipment")
    row["STOCK"] = product.get("stock")

    all_offers = [(tid, clean_cache[tid]) for tid in dict.fromkeys(offer_ids) if clean_cache.get(tid)]
    row["RATEPLAN DURATION (MONTHS)"] = get_months(product, all_offers)

    # line type gate
    offers = [(tid, ext) for tid, ext in all_offers if line_type in ext["eligible_line_types"]]

    hit = first_amount(offers, "device_discount", variant)
    if hit:
        row["UNIQUE DISCOUNT"] = fmt_money(hit[2]["value"])

    hit = first_amount(offers, "service_line_credit")
    if hit:
        row["SERVICE LINE OFFERS"] = hit[1]["cleaned_text"]
        row["SERVICE LINE OFFER $ VALUE"] = fmt_money(hit[2]["value"])

    hit = first_amount(offers, "switcher")
    if hit:
        row["SWITCHER"] = fmt_money(hit[2]["value"])
        row["SWITCHER_OFFER_TEXT"] = hit[1]["cleaned_text"]

    hit = first_amount(offers, "trade_in_max")
    if hit:
        row["Max TI"] = fmt_money(hit[2]["value"])
        row["Trade-In Value Type"] = hit[1].get("trade_in_value_type")

    # finance: new line ko welcome bonus bhi, upgrade ko sirf recurring
    one_time  = first_amount(offers, "finance_one_time", allow_monthly=True)
    recurring = first_amount(offers, "finance_recurring", allow_monthly=True)
    bill      = first_amount(offers, "finance_bill_discount", allow_monthly=True)
    parts = [h for h in ([one_time] if line_type == "New Line" else []) + [recurring, bill] if h]
    if parts:
        row["FINANCE OFFER $ VALUE"] = money_sum([h[2]["value"] for h in parts])
        if line_type == "New Line":
            row["FINANCE OFFER DESCRIPTION 1"] = parts[0][1]["cleaned_text"]
        else:
            row["FINANCE OFFER DESCRIPTION 1"] = "Recurring benefits only: " + " + ".join(
                fmt_money(h[2]["value"]) for h in parts)

    return row

In [27]:
clean_cache = {tid: clean_extraction(tid, ext) for tid, ext in cache.items()}

rows = [build_row(p, ids, lt)
        for p, ids in zip(products, product_offer_ids)
        for lt in LINE_TYPES]
df = pd.DataFrame(rows, columns=COLUMNS)

assert len(df) == 2 * len(products), "Two-row rule toota!"
assert list(df.columns) == COLUMNS, "Columns galat!"

df.to_csv("output.csv", index=False)
print("Saved output.csv:", len(df), "rows x", len(df.columns), "columns")
df[["PRODUCT NAME", "VARIANT", "LINE TYPE", "UNIQUE DISCOUNT", "SERVICE LINE OFFER $ VALUE",
    "SWITCHER", "Max TI", "FINANCE OFFER $ VALUE"]]

Saved output.csv: 24 rows x 29 columns


,PRODUCT NAME,VARIANT,LINE TYPE,UNIQUE DISCOUNT,SERVICE LINE OFFER $ VALUE,SWITCHER,Max TI,FINANCE OFFER $ VALUE
0,Apple iPhone 17 Pro Max,1 TB,New Line,$628.36,$200,$800,None,$450
1,Apple iPhone 17 Pro Max,1 TB,Upgrade,None,None,None,None,$250
2,Apple iPhone 17 Pro Max,2 TB,New Line,$812.36,$200,$800,None,$450
3,Apple iPhone 17 Pro Max,2 TB,Upgrade,None,None,None,None,$250
4,Apple iPhone 17 Pro,256 GB,New Line,$740.36,$200,$800,None,$450
5,Apple iPhone 17 Pro,256 GB,Upgrade,None,None,None,None,$250
6,Apple iPhone 17 Pro,512 GB,New Line,$724.36,$200,$800,None,$450
7,Apple iPhone 17 Pro,512 GB,Upgrade,None,None,None,None,$250
8,Apple iPhone 17 Pro,1 TB,New Line,$708.36,$200,$800,None,$450
9,Apple iPhone 17 Pro,1 TB,Upgrade,None,None,None,None,$250
